In [1]:
import json
from pathlib import Path
import pprint
from tqdm.notebook import tqdm
from collections import defaultdict,Counter


In [2]:
pp=pprint.PrettyPrinter(depth=4)
p=pprint.PrettyPrinter(depth=20)

In [3]:
path=Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/raw/ToolBench/data/data/answer/G1_answer/1_ChatGPT_DFS_woFilter_w2.json")
path2=Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/raw/ToolBench/data/data/answer/G2_answer/1_ChatGPT_DFS_woFilter_w2.json")
path3=Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/raw/ToolBench/data/data/answer/G3_answer/1_ChatGPT_DFS_woFilter_w2.json")
with open(path,"r",encoding="utf-8") as f:
    sample=json.load(f)
with open(path2,"r",encoding="utf-8") as f:
    sample2=json.load(f)
with open(path3,"r",encoding="utf-8") as f:
    sample3=json.load(f)
print(sample.keys())
print(sample2.keys())
print(sample3.keys())
print(type(sample))


dict_keys(['win', 'tree', 'forward_args', 'compare_candidates', 'answer_generation'])
dict_keys(['win', 'tree', 'forward_args', 'compare_candidates', 'answer_generation'])
dict_keys(['win', 'tree', 'forward_args', 'compare_candidates', 'answer_generation'])
<class 'dict'>


In [4]:
p.pprint(sample["win"])
p.pprint(sample2["win"])
p.pprint(sample3["win"])

True
True
False


In [5]:
print(sample["tree"].keys())
print(sample2["tree"].keys())   
print(sample3["tree"].keys())


dict_keys(['size', 'max_length', 'tree'])
dict_keys(['size', 'max_length', 'tree'])
dict_keys(['size', 'max_length', 'tree'])


In [6]:
node=sample["tree"]["tree"]
pp.pprint(node.keys())
print(len(node["children"]))
p.pprint(node)

dict_keys(['is_terminal', 'pruned', 'finished', 'depth', 'node_type', 'description', 'Elo', 'matching_time', 'child_count', 'expand_num', 'io_state', 'children'])
1
{'Elo': 0.0,
 'child_count': 1,
 'children': [{'Elo': 0.0,
               'child_count': 1,
               'children': [{'Elo': 0.0,
                             'child_count': 1,
                             'children': [{'Elo': 0.0,
                                           'child_count': 1,
                                           'children': [{'Elo': 0.0,
                                                         'child_count': 1,
                                                         'children': [{'Elo': 0.0,
                                                                       'child_count': 1,
                                                                       'children': [{'Elo': 0.0,
                                                                                     'child_count': 0,
                       

In [7]:
print(sample["answer_generation"].keys())
p.pprint(sample["answer_generation"])

dict_keys(['valid_data', 'query_count', 'forward_query_count', 'backward_query_count', 'total_tokens', 'final_answer', 'finish_type', 'function', 'chain', 'train_messages', 'query'])
{'backward_query_count': 0,
 'chain': [],
 'final_answer': '{\n'
                 '  "return_type": "give_answer",\n'
                 '  "final_answer": "The SQUAKE API is healthy and the list of '
                 'projects is available."\n'
                 '}',
 'finish_type': 'give_answer',
 'forward_query_count': 3,
 'function': [{'description': 'This is the subfunction for tool "squake", you '
                              'can use this tool.',
               'name': 'checkhealth_for_squake',
               'parameters': {'optional': [],
                              'properties': {},
                              'required': [],
                              'type': 'object'}},
              {'description': 'This is the subfunction for tool "squake", you '
                              'can use thi

In [8]:
p.pprint(sample["compare_candidates"])

[[{'Elo': 0.0,
   'child_count': 1,
   'depth': 1,
   'description': 'checkhealth_for_squake',
   'expand_num': 0,
   'finished': False,
   'is_terminal': False,
   'matching_time': 0,
   'node_type': 'Action',
   'pruned': False},
  {'Elo': 0.0,
   'child_count': 1,
   'depth': 2,
   'description': '{}',
   'expand_num': 1,
   'finished': False,
   'io_state': {},
   'is_terminal': False,
   'matching_time': 0,
   'node_type': 'Action Input',
   'observation': '{"error": "", "response": ""}',
   'observation_code': 0,
   'pruned': False},
  {'Elo': 0.0,
   'child_count': 1,
   'depth': 3,
   'description': 'projects_for_squake',
   'expand_num': 0,
   'finished': False,
   'is_terminal': False,
   'matching_time': 0,
   'node_type': 'Action',
   'pruned': False},
  {'Elo': 0.0,
   'child_count': 1,
   'depth': 4,
   'description': '{}',
   'expand_num': 2,
   'finished': True,
   'io_state': {},
   'is_terminal': False,
   'matching_time': 0,
   'node_type': 'Action Input',
   'observ

In [3]:
ROOT = Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/raw/ToolBench/data/data/answer")  

g1_files = list((ROOT / "G1_answer").glob("*.json"))
g2_files = list((ROOT / "G2_answer").glob("*.json"))
g3_files = list((ROOT / "G3_answer").glob("*.json"))


print(len(g1_files), len(g2_files), len(g3_files))

72783 29417 24286


In [10]:
def analyze(files):
    node_types = Counter()
    node_fields = defaultdict(set)
    observation_codes = Counter()
    max_depth = 0
    max_branching = 0
    def traverse(node):
        nonlocal max_depth, max_branching
        node_types[node["node_type"]] += 1
        node_fields[node["node_type"]].update(node.keys())
        if "observation_code" in node:
            observation_codes[node["observation_code"]] += 1
        max_depth = max(max_depth, node["depth"])
        max_branching = max(max_branching, len(node["children"]))
        for child in node["children"]:
            traverse(child)

    for file in files:
        with open(file, encoding="utf-8") as f:
            data = json.load(f)
        traverse(data["tree"]["tree"])

    return {
        "node_types": node_types,
        "node_fields": node_fields,
        "observation_codes": observation_codes,
        "max_depth": max_depth,
        "max_branching": max_branching,
    }

In [11]:
g1=analyze(g1_files)

In [12]:
g1

{'node_types': Counter({'Action Input': 1602436,
          'Action': 1529653,
          'Thought': 1086597}),
 'node_fields': defaultdict(set,
             {'Action Input': {'Elo',
               'child_count',
               'children',
               'depth',
               'description',
               'expand_num',
               'finished',
               'io_state',
               'is_terminal',
               'matching_time',
               'node_type',
               'observation',
               'observation_code',
               'pruned'},
              'Action': {'Elo',
               'child_count',
               'children',
               'depth',
               'description',
               'expand_num',
               'finished',
               'is_terminal',
               'matching_time',
               'node_type',
               'pruned'},
              'Thought': {'Elo',
               'child_count',
               'children',
               'depth',
               

In [13]:
g2=analyze(g2_files)
g2

{'node_types': Counter({'Action Input': 756726,
          'Action': 727309,
          'Thought': 471935}),
 'node_fields': defaultdict(set,
             {'Action Input': {'Elo',
               'child_count',
               'children',
               'depth',
               'description',
               'expand_num',
               'finished',
               'io_state',
               'is_terminal',
               'matching_time',
               'node_type',
               'observation',
               'observation_code',
               'pruned'},
              'Action': {'Elo',
               'child_count',
               'children',
               'depth',
               'description',
               'expand_num',
               'finished',
               'is_terminal',
               'matching_time',
               'node_type',
               'pruned'},
              'Thought': {'Elo',
               'child_count',
               'children',
               'depth',
               'de

In [14]:
g3=analyze(g3_files)
g3

{'node_types': Counter({'Action Input': 833867,
          'Action': 809581,
          'Thought': 513804}),
 'node_fields': defaultdict(set,
             {'Action Input': {'Elo',
               'child_count',
               'children',
               'depth',
               'description',
               'expand_num',
               'finished',
               'io_state',
               'is_terminal',
               'matching_time',
               'node_type',
               'observation',
               'observation_code',
               'pruned'},
              'Action': {'Elo',
               'child_count',
               'children',
               'depth',
               'description',
               'expand_num',
               'finished',
               'is_terminal',
               'matching_time',
               'node_type',
               'pruned'},
              'Thought': {'Elo',
               'child_count',
               'children',
               'depth',
               'de

In [15]:
for dataset, result in zip(
    ["G1", "G2", "G3"],
    [g1, g2, g3]
):

    print("="*60)
    print(dataset)
    print("="*60)

    for node_type, fields in result["node_fields"].items():

        print(node_type)

        for field in sorted(fields):
            print("   ", field)

        print()
        print(f"   Total {node_type}: {result['node_types'][node_type]}")
        print()

G1
Action Input
    Elo
    child_count
    children
    depth
    description
    expand_num
    finished
    io_state
    is_terminal
    matching_time
    node_type
    observation
    observation_code
    pruned

   Total Action Input: 1602436

Action
    Elo
    child_count
    children
    depth
    description
    expand_num
    finished
    is_terminal
    matching_time
    node_type
    pruned

   Total Action: 1529653

Thought
    Elo
    child_count
    children
    depth
    description
    expand_num
    finished
    is_terminal
    matching_time
    node_type
    observation_code
    pruned

   Total Thought: 1086597

G2
Action Input
    Elo
    child_count
    children
    depth
    description
    expand_num
    finished
    io_state
    is_terminal
    matching_time
    node_type
    observation
    observation_code
    pruned

   Total Action Input: 756726

Action
    Elo
    child_count
    children
    depth
    description
    expand_num
    finished
    is_termina

In [16]:
print("G1")
print(g1["observation_codes"])

print()

print("G2")
print(g2["observation_codes"])

print()

print("G3")
print(g3["observation_codes"])

G1
Counter({0: 776919, 4: 243995, 12: 233838, 11: 110453, 5: 70245, 3: 47453, 1: 28120, 7: 11647, 6: 6480, 2: 301, 9: 192, -1: 115, 8: 10})

G2
Counter({0: 478604, 5: 68958, 4: 58465, 11: 54203, 3: 21958, 12: 21404, 1: 14969, 6: 5220, 7: 3160, 2: 190, 8: 130, 9: 48})

G3
Counter({0: 397125, 12: 146728, 5: 122359, 4: 61032, 11: 46596, 1: 17553, 3: 14380, 6: 2967, 7: 706, 2: 135, -1: 7})


In [17]:
for name, result in zip(
    ["G1", "G2", "G3"],
    [g1, g2, g3]
):

    print(name)
    print("Max Depth     :", result["max_depth"])
    print("Max Branching :", result["max_branching"])
    print()

G1
Max Depth     : 14
Max Branching : 2

G2
Max Depth     : 14
Max Branching : 2

G3
Max Depth     : 14
Max Branching : 2



The Thought node also has an Observation code. Now we have to explore the examples to understand the semantics


In [18]:
def collect_observation_examples(files, max_examples=6):

    examples = defaultdict(list)

    def traverse(node):

        if "observation_code" in node:

            code = node["observation_code"]

            if len(examples[code]) < max_examples:

                examples[code].append({

                    "Node Type": node.get("node_type"),

                    "Description": node.get("description"),

                    "Observation": node.get("observation"),

                    "Finished": node.get("finished"),

                    "Pruned": node.get("pruned"),

                    "Terminal": node.get("is_terminal"),

                    "Depth": node.get("depth")

                })

        for child in node.get("children", []):
            traverse(child)

    for file in files:

        with open(file, encoding="utf-8") as f:
            data = json.load(f)

        traverse(data["tree"]["tree"])

    return examples

In [19]:
g1_examples = collect_observation_examples(g1_files)

In [20]:
for code in sorted(g1_examples):

    print(f"Observation Code : {code}")
    print()

    for i, example in enumerate(g1_examples[code], 1):

        print(f"\nExample {i}")

        p.pprint(example)

        print("-"*80)

Observation Code : -1


Example 1
{'Depth': 11,
 'Description': 'Error communicating with OpenAI: '
                "HTTPSConnectionPool(host='api.openai.com', port=443): Max "
                'retries exceeded with url: /v1/chat/completions (Caused by '
                "NewConnectionError('<urllib3.connection.HTTPSConnection "
                'object at 0x2c11a4130>: Failed to establish a new connection: '
                "[Errno 8] nodename nor servname provided, or not known'))",
 'Finished': False,
 'Node Type': 'Thought',
 'Observation': None,
 'Pruned': True,
 'Terminal': False}
--------------------------------------------------------------------------------

Example 2
{'Depth': 9,
 'Description': 'Error communicating with OpenAI: '
                "HTTPSConnectionPool(host='api.openai.com', port=443): Max "
                'retries exceeded with url: /v1/chat/completions (Caused by '
                "NewConnectionError('<urllib3.connection.HTTPSConnection "
                'obje

In [21]:
def print_tree(node, indent=""):

    node_type = node["node_type"]

    desc = node.get("description", "")

    desc = desc.replace("\n"," ")

    if len(desc) > 80:
        desc = desc[:80] + "..."

    print(f"{indent}{node_type} : {desc}")

    for child in node["children"]:
        print_tree(child, indent + "    ")

In [22]:
node=sample["tree"]["tree"] 
print_tree(node)

Action Input : 
    Action : checkhealth_for_squake
        Action Input : {}
            Action : projects_for_squake
                Action Input : {}
                    Action : Finish
                        Action Input : {   "return_type": "give_answer",   "final_answer": "The SQUAKE API is healthy a...


In [23]:
for file in g1_files[:2]:

    with open(file, "r", encoding="utf-8") as f:
        sample = json.load(f)

    root = sample["tree"]["tree"]

    print("-"*80)
    print(file.name)

    print("Root Node Type:", root["node_type"])
    p.pprint(root)
    print()

--------------------------------------------------------------------------------
53133_ChatGPT_DFS_woFilter_w2.json
Root Node Type: Action Input
{'Elo': 0.0,
 'child_count': 1,
 'children': [{'Elo': 0.0,
               'child_count': 1,
               'children': [{'Elo': 0.0,
                             'child_count': 1,
                             'children': [{'Elo': 0.0,
                                           'child_count': 1,
                                           'children': [{'Elo': 0.0,
                                                         'child_count': 1,
                                                         'children': [{'Elo': 0.0,
                                                                       'child_count': 1,
                                                                       'children': [{'Elo': 0.0,
                                                                                     'child_count': 0,
                                           

In [24]:
node=sample2["tree"]["tree"]
pp.pprint(node)

{'Elo': 0.0,
 'child_count': 1,
 'children': [{'Elo': 0.0,
               'child_count': 1,
               'children': [{...}],
               'depth': 1,
               'description': 'get_tracking_data_for_create_container_tracking',
               'expand_num': 0,
               'finished': False,
               'is_terminal': False,
               'matching_time': 0,
               'node_type': 'Action',
               'pruned': False}],
 'depth': 0,
 'description': '',
 'expand_num': 0,
 'finished': False,
 'io_state': {},
 'is_terminal': False,
 'matching_time': 0,
 'node_type': 'Action Input',
 'pruned': False}


In [6]:
def count_successful_finishes(node):
    """
    Counts Finish -> Action Input(observation_code=3) occurrences
    in the subtree rooted at 'node'.
    """

    count = 0

    if node["node_type"] == "Action":
        if node["description"] == "Finish":

            if len(node["children"]) > 0:

                child = node["children"][0]

                if (
                    child["node_type"] == "Action Input"
                    and child.get("observation_code") == 3
                ):
                    count += 1

    for child in node["children"]:
        count += count_successful_finishes(child)

    return count

In [7]:
finish_counter = Counter()
for file in tqdm(g1_files):

    with open(file, "r", encoding="utf-8") as f:
        sample = json.load(f)

    root = sample["tree"]["tree"]

    n = count_successful_finishes(root)

    finish_counter[n] += 1

print(finish_counter)

  0%|          | 0/72783 [00:00<?, ?it/s]

Counter({1: 47453, 0: 25330})


In [8]:
finish_counter = Counter()
for file in tqdm(g2_files):

    with open(file, "r", encoding="utf-8") as f:
        sample = json.load(f)

    root = sample["tree"]["tree"]

    n = count_successful_finishes(root)

    finish_counter[n] += 1

print(finish_counter)

  0%|          | 0/29417 [00:00<?, ?it/s]

Counter({1: 21928, 0: 7479, 3: 10})


In [9]:
finish_counter = Counter()
for file in tqdm(g3_files):

    with open(file, "r", encoding="utf-8") as f:
        sample = json.load(f)

    root = sample["tree"]["tree"]

    n = count_successful_finishes(root)

    finish_counter[n] += 1

print(finish_counter)

  0%|          | 0/24286 [00:00<?, ?it/s]

Counter({1: 14380, 0: 9906})
